In [1]:
import numpy as np
import matplotlib.pyplot as plt

from troma import data_structure as ds
from troma import ConstraintSketch
from troma import bind_optimizer, get_optimizer
from troma import matching_pursuit

In [2]:
#example 2
number_spins = 6

#abstract
spectrum_bin = [
    [0,0,0,0,0,0],
    [0,1,0,1,0,1],
    [0,0,1,0,0,1],
    [1,1,1,1,1,1]
]
spectrum_pos = [ds.dit_string_to_integer(s) for s in spectrum_bin]
spectrum_val = [0.5,5,3,2]

#explicit
full_spectrum = np.zeros((2**number_spins,))
full_spectrum[spectrum_pos] = spectrum_val

#Define marginals and sketch
interaction_size = 2
constraints = ConstraintSketch.build_nearest_neighbors_sketch(number_spins, interaction_size, 2)
y = ConstraintSketch.compute_marginal((spectrum_bin, spectrum_val), constraints)

In [3]:
#Quantum methods

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2, Session

backend = AerSimulator()
sampler = SamplerV2(mode=backend)

opti = bind_optimizer("qaoa", sampler=sampler,number_shots=4096)
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=2, step=None, interaction_size=interaction_size, optimizer=opti))

[[21.    5.6 ]
 [ 9.    3.08]]


In [8]:
#Quantum methods
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from troma.optimization.quantum_cost import estimate_matching_pursuit_qpu_cost

service = QiskitRuntimeService()
backend = service.backend("ibm_marrakesh")

# with Session(backend=backend, max_time="5m") as session:

sampler = SamplerV2(mode=backend)

opti = bind_optimizer("qaoa", sampler=sampler,number_shots=4096, number_layers=4, method="COBYLA", optimizer_options={"maxiter": 10, "maxfev": 15})

estimate = estimate_matching_pursuit_qpu_cost(
    constraints,
    y,
    bit_string_length=number_spins,
    optimizer=opti,
    matching_pursuit_iterations=1,
)

qiskit_runtime_service.__init__:WARNING:2026-04-22 16:10:40,701: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (premium), the available account instances are: S0135 Quaranea. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-22 16:10:40,702: Using instance: S0135 Quaranea, plan: premium


Estimated circuits per QAOA run: 11
  Estimated duration per circuit: 3.000 s
  Estimated quantum time per QAOA run: 33.000 s
  Estimated quantum time for full matching pursuit: 33.000 s


In [ ]:
#Quantum methods

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2, Session

service = QiskitRuntimeService()
backend = service.backend("ibm_marrakesh")

# with Session(backend=backend, max_time="5m") as session:

sampler = SamplerV2(mode=backend)
sampler.options.max_execution_time = 3

opti = bind_optimizer("qaoa", sampler=sampler,number_shots=512, number_layers=2, method="COBYLA", optimizer_options={"maxiter": 10, "maxfev": 15})
print(matching_pursuit("abstract", y, constraints, number_spins, iteration_number=1, step=None, interaction_size=interaction_size, optimizer=opti))

qiskit_runtime_service.__init__:WARNING:2026-04-20 19:01:12,476: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (premium), the available account instances are: S0135 Quaranea. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-20 19:01:12,477: Using instance: S0135 Quaranea, plan: premium
/home/baptiste/Gdrive/Projects/troma_lib/src/troma/optimization/quantum.py:188: OptimizeWarning: Unknown solver options: maxfev
  res = sk_opt.minimize(


[[21.   5.6]]


In [16]:
job = service.job("d679945bujdc73cvldsg")
job.metrics()
job.usage_estimation

{'quantum_seconds': 3.062339017}